# Why Higher F1 is Better (Not Lower)

> **Common Question**: "F1 balances precision and recall, but what does 'balance' really mean?"

## What "Balance" Actually Means

Balance refers to how **equal** precision and recall are. The F1 score is highest when:
- Both precision AND recall are high
- They are relatively close to each other

The score decreases when:
- One metric is significantly lower than the other
- Either metric is poor

# Why F1 and Accuracy Can Be Inverse

## The Simple Math Reason

**Accuracy includes True Negatives. F1 ignores them completely.**

This is the entire reason they can move in opposite directions.

---

## The Classic Inverse Case
> Dataset: 100 items (95 negative, 5 positive)

### Model 1: "Always Predict Negative"

| | Predicted Neg | Predicted Pos |
|---|---|---|
| **Actual Neg** | 95 (TN) | 0 (FP) |
| **Actual Pos** | 5 (FN) | 0 (TP) |
> Accuracy = (95 + 0) / 100 = 95%; F1 = 0 (because precision or recall = 0)

### Model 2: "Actually Tries"

| | Predicted Neg | Predicted Pos |
|---|---|---|
| **Actual Neg** | 90 (TN) | 5 (FP) |
| **Actual Pos** | 1 (FN) | 4 (TP) |
> Accuracy = (90 + 4) / 100 = 94% (LOWER than Model 1); F1 = 2 × (0.44 × 0.8) / (0.44 + 0.8) = 0.57 (HIGHER than Model 1)

---

## The Only Time They're Inverse

They are inverse when:
1. Dataset is **highly imbalanced** (mostly negatives)
2. You start from a **"predict majority class"** baseline
3. Improving F1 **requires sacrificing some True Negatives** (converting them to False Positives)

---

## The Takeaway

They're inverse ONLY when:
- ✅ You care about finding the rare positive class
- ✅ This forces you to misclassify some negatives
- ✅ Those negatives were boosting your accuracy before

Otherwise, they usually move together.

# Model Evaluation Metrics: R², Log-Likelihood, AIC, BIC

## 1. R² (Coefficient of Determination)

**What it measures**: How much of the variance in the target variable your model explains.

**Formula**:
$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

**Simple Explanation**: 
- It tells you what percentage of the data's variation your model captures
- Like saying "my model explains 80% of why the data moves around"

**Strengths**:
- ✅ Intuitive (0-1 scale, like a percentage)
- ✅ Easy to communicate to non-technical people
- ✅ No units, so comparable across different datasets

**Weaknesses**:
- ❌ **ALWAYS increases** when you add more predictors (even random noise)
- ❌ Can't compare models with different datasets
- ❌ Doesn't tell you if coefficients are significant

---

## 2. Log-Likelihood

**What it measures**: The probability of observing your data given your model parameters.

**Formula**:
$$\text{Log-Likelihood} = \sum_{i=1}^{n} \log(P(y_i | x_i, \theta))$$

**Simple Explanation**: 
- How likely is it that this model produced the data we see?
- Like asking "what are the odds my model is the real truth?"

**Strengths**:
- ✅ Foundation for many other metrics (AIC, BIC come from it)
- ✅ Works for any model type (regression, classification, etc.)
- ✅ Directly compares probability of data under different models

**Weaknesses**:
- ❌ Always negative (confuses beginners)
- ❌ No absolute interpretation (only useful for comparison)
- ❌ Can be astronomically large/small numbers

---

## 3. AIC (Akaike Information Criterion)

**What it measures**: Model quality with a penalty for complexity.

**Formula**:
$$AIC = -2\log(L) + 2k$$

**Simple Explanation**: 
- It's like log-likelihood but with a "complexity tax"
- Every time you add a parameter, you pay a small penalty (2)

**Strengths**:
- ✅ Prevents overfitting by penalizing extra parameters
- ✅ Theoretically grounded in information theory
- ✅ Good for prediction-focused model selection

**Weaknesses**:
- ❌ Numbers are arbitrary (only differences matter)
- ❌ Penalty might be too small for very large datasets
- ❌ Assumes certain statistical properties (large sample, etc.)

---

## 4. BIC (Bayesian Information Criterion)

**What it measures**: Similar to AIC but with a stricter penalty for complexity.

**Formula**:
$$BIC = -2\log(L) + k\log(n)$$

**Simple Explanation**: 
- Same as AIC but the "complexity tax" gets bigger with more data
- log(n) means: with 1000 data points, penalty = k × 6.9 (vs AIC's k × 2)

**Strengths**:
- ✅ Much stricter on complexity (especially with big data)
- ✅ Consistent: will pick the true model if it exists (as n → ∞)
- ✅ Better for explanation-focused modeling

**Weaknesses**:
- ❌ Can be TOO strict, underfitting with small samples
- ❌ Assumes the true model is among candidates
- ❌ Numbers even less interpretable than AIC

---

## Quick Analogy

| Metric | Analogy |
|--------|---------|
| **R²** | Your grade on a test (0-100%) |
| **Log-Likelihood** | How sure you are about each answer |
| **AIC** | Test score minus 2 points per cheat sheet used |
| **BIC** | Test score minus (log of questions) points per cheat sheet |

---

## When They Disagree (Common Scenario)
| Criterion | Prefers | Why |
|-----------|---------|-----|
| R² |	Complex model | Always picks the one with more predictors |
| Log-Likelihood |Complex model | Better fit to training data |
| AIC | Medium model | Penalty balances fit vs complexity |
| BIC | Simple model | Stricter penalty favors parsimony |

> Parcimony = the principle of favoring the simplest model with the fewest parameters that adequately explains the data

---

## Rule of thumb:
- Use AIC if you care about prediction
- Use BIC if you care about finding the "true" model
- Use R² only for explaining to non-technical people
- Use Log-Likelihood as input to the others, not alone

### Sample Code for $R^2$, Log-Likelihood, AIC, BIC

In [3]:
import statsmodels.api as sm
import numpy as np
import pandas as pd
np.random.seed(42)

# Create data
X = np.linspace(0, 10, 100)
y = 1 + 2*X + np.random.normal(0, 3, 100)
X = sm.add_constant(X)

# Create model
model = sm.OLS(y, X).fit()

# Get metrics
print("R²:", model.rsquared)
print("Log-Likelihood:", model.llf)
print("AIC:", model.aic)
print("BIC:", model.bic)

# Or just look at the summary
print("\n", model.summary())

R²: 0.8284917657651463
Log-Likelihood: -241.52087776079046
AIC: 487.0417555215809
BIC: 492.2520958935571

                             OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.828
Model:                            OLS   Adj. R-squared:                  0.827
Method:                 Least Squares   F-statistic:                     473.4
Date:                Tue, 03 Mar 2026   Prob (F-statistic):           2.66e-39
Time:                        15:44:18   Log-Likelihood:                -241.52
No. Observations:                 100   AIC:                             487.0
Df Residuals:                      98   BIC:                             492.3
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------